In [ ]:
import os
import urllib.request
import zipfile
import glob
import math
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from scipy import signal
import matplotlib.pyplot as plt

from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score, precision_recall_curve, auc

!pip install torchinfo
import torchinfo

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)
np.random.seed(0)

#Data Preprocessing for UCI HAR (benign) and SisFall (anomaly)

# PART 1: DATA PREPARATION (UCI HAR NORMAL DATA)
## Step 1: Download & Extract UCI HAR

In [ ]:
DATASET_URL = "https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip"
ZIP_PATH = "har_dataset.zip"
EXTRACT_DIR = "uci_har_data"

if not os.path.exists(EXTRACT_DIR):
    print("Downloading UCI HAR dataset...")
    urllib.request.urlretrieve(DATASET_URL, ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)

    inner_zip = os.path.join(EXTRACT_DIR, "UCI HAR Dataset.zip")
    if os.path.exists(inner_zip):
        with zipfile.ZipFile(inner_zip, 'r') as zip_ref:
            zip_ref.extractall(EXTRACT_DIR)
    print("UCI HAR Ready!")
else:
    print("UCI HAR already downloaded.")

UCI HAR Ready!


## Step 2: Load & Resample UCI HAR

In [ ]:
BASE_PATH = os.path.join(EXTRACT_DIR, "UCI HAR Dataset")

# Reduced to 6 channels (accel x,y,z + gyro x,y,z) instead of 9. We use
# total_acc (NOT body_acc) because it still includes the gravity component,
# which matches SisFall's raw, gravity-inclusive accelerometer stream far
# more closely than the gravity-removed body_acc would. This also lines the
# channel count up with the 6 SisFall channels we keep after conversion
# (see Part 2), instead of arbitrarily matching 9-to-9.
SIGNAL_NAMES = [
    "total_acc_x", "total_acc_y", "total_acc_z",
    "body_gyro_x", "body_gyro_y", "body_gyro_z"
]

def load_har_signals(mode="train"):
    signals_list = []
    for sig in SIGNAL_NAMES:
        file_path = os.path.join(BASE_PATH, mode, "Inertial Signals", f"{sig}_{mode}.txt")
        data = np.loadtxt(file_path)
        signals_list.append(data)
    return np.transpose(np.array(signals_list), (1, 2, 0))

def preprocess_and_window(X_raw, orig_fs=50, target_fs=100, window_sec=2.0):
    num_samples, orig_len, num_channels = X_raw.shape
    resampled_len = int(orig_len * (target_fs / orig_fs))
    target_samples = int(window_sec * target_fs)

    X_resampled = np.zeros((num_samples, resampled_len, num_channels))
    for i in range(num_samples):
        for c in range(num_channels):
            X_resampled[i, :, c] = signal.resample(X_raw[i, :, c], resampled_len)

    center_idx = resampled_len // 2
    half_window = target_samples // 2
    return X_resampled[:, center_idx - half_window : center_idx + half_window, :]

X_train_100hz = preprocess_and_window(load_har_signals("train"))
X_test_100hz = preprocess_and_window(load_har_signals("test"))

## Step 2.5: Download & Extract SisFall Dataset
* Note: The official SisFall download link is broken. We use `gdown` to pull the dataset from a recognized public Google Drive mirror.

In [ ]:
# Install gdown directly from the notebook
#!pip install gdown

import os
import zipfile
import gdown

SISFALL_ZIP = "SisFall.zip"
SISFALL_DIR = "./SisFall_dataset"

# We check if the directory already exists to prevent re-downloading on multiple runs
if not os.path.exists(SISFALL_DIR):
    print("Downloading SisFall dataset from Google Drive mirror...")

    # Using a known public Google Drive mirror for the SisFall Dataset
    file_id = '1-E-TLd5_J-DDWZXkuYL-moMpoezlMn4Z'
    url = f'https://drive.google.com/uc?id={file_id}'

    # Download the zip file
    gdown.download(url, SISFALL_ZIP, quiet=False)

    print("Extracting SisFall dataset...")
    try:
        # Extract the contents into our target directory
        with zipfile.ZipFile(SISFALL_ZIP, 'r') as zip_ref:
            zip_ref.extractall(SISFALL_DIR)
        print("SisFall dataset downloaded and extracted successfully!")
    except zipfile.BadZipFile:
        print("Error: The downloaded file is not a valid zip archive. Please verify your connection or gdown installation.")
else:
    print("SisFall dataset already exists locally. Skipping download.")

Downloading...
From (original): https://drive.google.com/uc?id=1-E-TLd5_J-DDWZXkuYL-moMpoezlMn4Z
From (redirected): https://drive.google.com/uc?id=1-E-TLd5_J-DDWZXkuYL-moMpoezlMn4Z&confirm=t&uuid=b88733ca-3dfa-40b1-9ee5-c11f8cf4d179
To: /content/SisFall.zip
100%|██████████| 226M/226M [00:04<00:00, 46.0MB/s]


Extracting SisFall dataset...
SisFall dataset downloaded and extracted successfully!


# PART 2: DATA PREPARATION (SISFALL ANOMALY DATA)
## Step 3: Parse, Resample, and Window SisFall
* SisFall natively records at 200 Hz. We resample to 100 Hz.
* We specifically target Fall files (usually starting with 'F').

In [ ]:
# --- Sensor conversion constants (raw ADC counts -> physical units) ---
# SisFall raw columns per line: [ADXL345 x,y,z, ITG3200 x,y,z, MMA8451Q x,y,z]
# We only keep the first accelerometer + the gyroscope (6 channels), converted
# to physical units, so they are on the same footing as UCI HAR's total_acc
# (g) and body_gyro (rad/s) rather than raw uncalibrated integer counts.
ACCEL_RANGE_G = 16          # ADXL345 configured range: +/-16 g
ACCEL_RES_BITS = 13         # ADXL345 ADC resolution
GYRO_RANGE_DPS = 2000       # ITG3200 configured range: +/-2000 deg/s
GYRO_RES_BITS = 16          # ITG3200 ADC resolution

ACCEL_SCALE_G = (2 * ACCEL_RANGE_G) / (2 ** ACCEL_RES_BITS)                    # g per LSB
GYRO_SCALE_RAD = (2 * GYRO_RANGE_DPS) / (2 ** GYRO_RES_BITS) * (np.pi / 180)   # rad/s per LSB


def load_sisfall_anomalies(sisfall_dir, target_fs=100, window_sec=2.0):
    orig_fs = 200
    target_samples = int(window_sec * target_fs)

    # Grab all fall files
    fall_files = glob.glob(os.path.join(sisfall_dir, '**', 'F*.txt'), recursive=True)

    if not fall_files:
        raise FileNotFoundError(f"No SisFall anomaly files found in {sisfall_dir}.")

    windows = []
    n_files_used = 0
    print(f"Processing {len(fall_files)} SisFall anomaly files...")

    error_printed = False
    for f in fall_files:
        try:
            # --- BULLETPROOF PURE PYTHON PARSING ---
            file_data = []
            with open(f, 'r') as file:
                for line in file:
                    # Strip whitespace, newlines, semicolons, and trailing commas from the line
                    cleaned_line = line.strip().rstrip(';').rstrip(',')
                    if not cleaned_line:
                        continue

                    # Split the line by commas
                    parts = cleaned_line.split(',')

                    # We only care about lines that contain all 9 raw sensor readings
                    if len(parts) >= 9:
                        try:
                            row_raw = [float(p) for p in parts[:9]]
                            # Convert accel (cols 0:3, ADXL345) and gyro (cols 3:6, ITG3200)
                            # to physical units. Drop the second accelerometer (cols 6:9,
                            # MMA8451Q) - redundant, and not something we have a matching
                            # channel for on the UCI HAR side.
                            accel = [v * ACCEL_SCALE_G for v in row_raw[0:3]]
                            gyro = [v * GYRO_SCALE_RAD for v in row_raw[3:6]]
                            file_data.append(accel + gyro)
                        except ValueError:
                            # If a specific line has text/garbage instead of numbers, skip just that line
                            continue

            data = np.array(file_data)

            # If the file ended up empty or malformed, skip it
            if data.size == 0 or data.shape[1] != 6:
                continue
            # ---------------------------------------

            # Resample from 200Hz to 100Hz
            resampled_len = int(len(data) * (target_fs / orig_fs))
            data_resampled = signal.resample(data, resampled_len, axis=0)

            # Slice into non-overlapping 2-second candidate windows, but only KEEP
            # the one with the peak acceleration magnitude for this file. A 15s
            # fall trial contains a genuine impact for well under 2s of its total
            # length - labeling every window in the file "anomaly" mislabels the
            # normal standing/walking segments before and after the fall.
            num_windows = len(data_resampled) // target_samples
            if num_windows == 0:
                continue

            best_window = None
            best_score = -np.inf
            for i in range(num_windows):
                start = i * target_samples
                end = start + target_samples
                w = data_resampled[start:end, :]
                accel_mag = np.linalg.norm(w[:, 0:3], axis=1)
                peak = accel_mag.max()
                if peak > best_score:
                    best_score = peak
                    best_window = w

            windows.append(best_window)
            n_files_used += 1

        except Exception as e:
            if not error_printed:
                print(f"Error parsing file {f}: {e}")
                error_printed = True
            continue

    windows = np.array(windows)

    # Quick sanity check: since we downloaded this from an unofficial Google
    # Drive mirror (the official SisFall link is down), confirm the parsed
    # values land in physically plausible ranges rather than silently trusting
    # a possibly-corrupted download.
    print(f"Files successfully parsed         : {n_files_used} / {len(fall_files)}")
    print(f"Anomaly windows generated (1/file): {windows.shape[0]}")
    if windows.shape[0] > 0:
        print(f"Sanity check - accel range (g)    : [{windows[:, :, 0:3].min():.2f}, {windows[:, :, 0:3].max():.2f}]  (expect within +/-{ACCEL_RANGE_G}g)")
        print(f"Sanity check - gyro range (rad/s) : [{windows[:, :, 3:6].min():.2f}, {windows[:, :, 3:6].max():.2f}]  (expect within +/-{GYRO_RANGE_DPS * np.pi / 180:.1f} rad/s)")

    return windows

# Load the actual SisFall data
X_sisfall_all = load_sisfall_anomalies(SISFALL_DIR)
print(f"Total SisFall windows generated: {X_sisfall_all.shape[0]}")


Processing 1744 SisFall anomaly files...
Files successfully parsed         : 1744 / 1744
Anomaly windows generated (1/file): 1744
Sanity check - accel range (g)    : [-16.26, 15.97]  (expect within +/-16g)
Sanity check - gyro range (rad/s) : [-33.19, 36.03]  (expect within +/-34.9 rad/s)
Total SisFall windows generated: 1744


## Step 4: Build the 80/20 Test Pool

In [ ]:
# 1. Held-out validation split from the officially-"train" HAR normal data.
#    Used ONLY to calibrate the anomaly threshold below - never for gradient
#    updates, and never overlapping with the test set.
n_total_train = X_train_100hz.shape[0]
n_val = int(n_total_train * 0.10)
perm = np.random.permutation(n_total_train)
val_idx, fit_idx = perm[:n_val], perm[n_val:]

X_fit_tensor = torch.tensor(X_train_100hz[fit_idx], dtype=torch.float32)
X_val_tensor = torch.tensor(X_train_100hz[val_idx], dtype=torch.float32)

# 2. Test Set Math: 50% of Normal count = exactly ??% of the combined pool
n_normal_test = X_test_100hz.shape[0]
n_anomaly_test = (n_normal_test * 1) #Equal parts normal and anomaly
if X_sisfall_all.shape[0] < n_anomaly_test:
    print(f"Warning: Not enough SisFall data ({X_sisfall_all.shape[0]}) to hit exact 50%. Using all available.")
    print(f"Resizing normal dataset size to match all existing anomaly dataset")
    n_anomaly_test = X_sisfall_aill.shape[0]
    X_test_100hz = np.resize(X_test_100hz, X_sisfall_all.shape)
# Randomly select the exact number of anomalies needed (seeded above)
indices = np.random.choice(X_sisfall_all.shape[0], n_anomaly_test, replace=False)
X_anomaly_real = X_sisfall_all[indices]

# Stack Test Data & Labels
X_test_combined = np.concatenate([X_test_100hz, X_anomaly_real], axis=0)
y_test_labels = np.concatenate([np.zeros(len(X_test_100hz)), np.ones(n_anomaly_test)], axis=0)

X_test_tensor = torch.tensor(X_test_combined, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_labels, dtype=torch.float32)

# =====================================================================
# 3. FEATURE STANDARDIZATION (Z-Score Normalization)
# Compute channel mean & std ONLY on training fit pool to prevent leakage
# Assumes tensor shape: [Batch_Size, Time_Steps, Channels]
# =====================================================================
fit_mean = X_fit_tensor.mean(dim=(0, 1), keepdim=True)
fit_std = X_fit_tensor.std(dim=(0, 1), keepdim=True) + 1e-8

# Apply scaling using fit statistics across all sets
X_fit_tensor = (X_fit_tensor - fit_mean) / fit_std
X_val_tensor = (X_val_tensor - fit_mean) / fit_std
X_test_tensor = (X_test_tensor - fit_mean) / fit_std
# =====================================================================

BATCH_SIZE = 64
train_loader = DataLoader(TensorDataset(X_fit_tensor), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_tensor), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(TensorDataset(X_test_tensor, y_test_tensor), batch_size=BATCH_SIZE, shuffle=False)

print(f"Fit Pool (Normal, trained on)     : {len(X_fit_tensor)} samples")
print(f"Val Pool (Normal, threshold only) : {len(X_val_tensor)} samples")
print(f"Test Pool Size Percentage         : {(len(X_test_tensor) / (len(X_fit_tensor) + len(X_test_tensor))):.2f}%")
print(f"Test Pool Total                   : {len(X_test_tensor)} samples")
print(f"  |-- Normal (UCI HAR)   : {np.sum(y_test_labels == 0)}")
print(f"  \\-- Anomaly (SisFall)  : {np.sum(y_test_labels == 1)} ({np.mean(y_test_labels==1)*100:.1f}%)")

Resizing normal dataset size to match all existing anomaly dataset
Fit Pool (Normal, trained on)     : 6617 samples
Val Pool (Normal, threshold only) : 735 samples
Test Pool Size Percentage         : 0.35%
Test Pool Total                   : 3488 samples
  |-- Normal (UCI HAR)   : 1744
  \-- Anomaly (SisFall)  : 1744 (50.0%)


#Transformer Model
## [Based on this paper](https://www.sciencedirect.com/science/article/pii/S0952197623001483)

###The paper's model consisted of 8 heads, 3 layers, and a feature size of 256, Conv1d was 1x3, 2 fc layers in each encoder layer, 2 fc layers in decoder

### I think the data was 6 channels wide (accel XYZ and gyro XYZ)

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.encoding = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        self.encoding[:, 0::2] = torch.sin(position * div_term)
        self.encoding[:, 1::2] = torch.cos(position * div_term)
        self.encoding = self.encoding.unsqueeze(0).to(device)

    def forward(self, x):
        return x + self.encoding[:, :x.size(1)].detach()

class TransformerEncoderLayer(nn.Module):
  #Individual layer based on the pape
  # Create Q,K,V values from each embedding, add them with embedding, feed them to fc layer,
  # and add with residual again
  def __init__(self, hidden_size, n_heads, hidden_fc, dropout):
    super().__init__()
    self.self_attn = nn.MultiheadAttention(
        hidden_size, n_heads, dropout, batch_first = True
    )
    self.norm1 = nn.LayerNorm(hidden_size)
    self.fc1 = nn.Sequential(
        nn.Linear(hidden_size, hidden_fc),
        nn.ReLU(),
        nn.Linear(hidden_fc, hidden_size)
    )
    self.norm2 = nn.LayerNorm(hidden_size)
    self.dropout = nn.Dropout(dropout)

  def forward(self, x, attn_mask):
    #x,x,x creates Q,K, and V values for each input x
    attn_out, _ = self.self_attn(x,x,x, attn_mask=attn_mask, need_weights = False)
    x = self.norm1(x + self.dropout(attn_out))

    ff_out = self.fc1(x)
    x = self.norm2(x + self.dropout(ff_out))
    return x

class TransformerEncoderStack(nn.Module):
  def __init__(self, hidden_size, n_heads, hidden_fc, n_layers, dropout = 0.1):
    super().__init__()
    self.layers = nn.ModuleList(
        [TransformerEncoderLayer(hidden_size, n_heads, hidden_fc, dropout) for _ in range(n_layers)]
    )
    self.n_layers = n_layers

  def forward(self, x: torch.Tensor, attn_mask: torch.Tensor = None):
        layer_outputs = []
        for layer in self.layers:
            x = layer(x, attn_mask=attn_mask)
            layer_outputs.append(x)  # (batch, L, d_model) each
        return layer_outputs  # list of length N

class ForecastDecoder(nn.Module):
  def __init__(
      self,
      hidden_size,
      n_layers,
      window_size,
      forecast_horizon,
      conv_channels = 32,
      kernel_size = 3,
      hidden_fc = 256,
      dropout = 0.1,
      n_channels = 6
  ):
    super().__init__()
    self.n_layers = n_layers
    self.hidden_size = hidden_size
    self.forecast_horizon = forecast_horizon
    self.n_channels = n_channels

    self.conv1d = nn.Conv1d(
        in_channels= n_layers * hidden_size,
        out_channels= conv_channels,
        kernel_size = kernel_size,
        padding = kernel_size // 2
    )
    self.activation = nn.ReLU()

    self.fc = nn.Sequential(
        nn.Linear(conv_channels * window_size, hidden_fc),
        nn.ReLU(),
        nn.Dropout(dropout),
        nn.Linear(hidden_fc, forecast_horizon * n_channels)
    )

  def forward(self, layer_outputs: list):
    batch_size, L, _ = layer_outputs[0].shape

    stacked = torch.stack(layer_outputs, dim = 1)
    stacked = stacked.permute(0, 1, 3, 2).reshape(batch_size, self.n_layers * self.hidden_size, L)

    conv_out = self.activation(self.conv1d(stacked))
    conv_out = conv_out.flatten(start_dim=1)

    forecast = self.fc(conv_out) # (batch, horizon*channels)
    forecast = forecast.view(batch_size, self.forecast_horizon, self.n_channels)
    return forecast

class TransformerForecaster(nn.Module):
  def __init__(
        self,
        input_size: int,
        hidden_size: int,
        window_size: int = 50,
        forecast_horizon = 50,
        n_heads: int = 8,
        n_layers: int = 3,
        conv_channels = 32,
        fc_layers = 2,
        kernel_size = 3,
        hidden_fc = 256,
        dropout: float = 0.1,
        use_pos_encoding = False
      ):
    super().__init__()

    self.input_proj = nn.Linear(input_size, hidden_size)
    self.pos_encoding = (
        PositionalEncoding(hidden_size, max_len = window_size)
        if use_pos_encoding else None
      )
    self.encoder = TransformerEncoderStack(hidden_size, n_heads, hidden_fc, n_layers, dropout)
    self.decoder = ForecastDecoder(hidden_size, n_layers, window_size, forecast_horizon, conv_channels, kernel_size, hidden_fc, dropout, n_channels=input_size)

  @staticmethod
  def _generate_causal_mask(size: int, device) -> torch.Tensor:
      """Upper-triangular boolean mask so position i cannot attend to j > i
      (the 'masked' in Masked Multi-Head Attention)."""
      return torch.triu(torch.ones(size, size, device=device, dtype=torch.bool), diagonal=1)

  def forward(self, x: torch.Tensor) -> torch.Tensor:
      # x: (batch, L, input_dim)
      batch_size, L, _ = x.shape
      device = x.device

      h = self.input_proj(x) # (batch, L, d_model)
      if self.pos_encoding is not None:
        h = self.pos_encoding(h)
      causal_mask = self._generate_causal_mask(L, device)
      layer_outputs = self.encoder(h, attn_mask=causal_mask)  # list of N x (batch, L, d_model)

      forecast = self.decoder(layer_outputs)  # (batch, forecast_horizon, n_channels)
      return forecast


#GRU Model





In [ ]:
class GRUEncoderLayer(nn.Module):
  def __init__(self, hidden_size, hidden_fc, dropout):
    super().__init__()
    self.gru = nn.GRU(hidden_size, hidden_size, num_layers=1, batch_first=True)
    self.norm1 = nn.LayerNorm(hidden_size)
    self.fc1 = nn.Sequential(
        nn.Linear(hidden_size, hidden_fc),
        nn.ReLU(),
        nn.Linear(hidden_fc, hidden_size)
    )
    self.norm2 = nn.LayerNorm(hidden_size)
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    gru_out, _ = self.gru(x)
    x = self.norm1(x + self.dropout(gru_out))

    ff_out = self.fc1(x)
    x = self.norm2(x + self.dropout(ff_out))
    return x

class GRUEncoderStack(nn.Module):
  def __init__(self, hidden_size, hidden_fc, n_layers, dropout = 0.1):
    super().__init__()
    self.layers = nn.ModuleList(
        [GRUEncoderLayer(hidden_size, hidden_fc, dropout) for _ in range(n_layers)]
    )
    self.n_layers = n_layers

  def forward(self, x: torch.Tensor):
    layer_outputs = []
    for layer in self.layers:
      x = layer(x)
      layer_outputs.append(x)
    return layer_outputs

class GRUForecaster(nn.Module):
  def __init__(
        self,
        input_size: int,
        hidden_size: int,
        window_size: int = 50,
        forecast_horizon = 50,
        n_heads: int = 8,
        n_layers: int = 3,
        conv_channels = 32,
        fc_layers = 2,
        kernel_size = 3,
        hidden_fc = 256,
        dropout: float = 0.1,
        use_pos_encoding = False,
      ):
    super().__init__()

    self.input_proj = nn.Linear(input_size, hidden_size)
    self.pos_encoding = (
        PositionalEncoding(hidden_size, max_len=window_size)
        if use_pos_encoding else None
    )
    self.encoder = GRUEncoderStack(hidden_size, hidden_fc, n_layers, dropout)
    self.decoder = ForecastDecoder(
        hidden_size, n_layers, window_size, forecast_horizon, conv_channels,
        kernel_size, hidden_fc, dropout, n_channels=input_size
    )
  def forward(self, x: torch.Tensor) -> torch.Tensor:
     h = self.input_proj(x) # (batch, L, d_model)
     if self.pos_encoding is not None:
         h = self.pos_encoding(h)
     layer_outputs = self.encoder(h)  # list of N x (batch, L, d_model)
     forecast = self.decoder(layer_outputs)  # (batch, forecast_horizon, n_channels)
     return forecast




In [ ]:
window_size = 200
forecast_horizon = 1
learning_rate = 0.001

model = TransformerForecaster(    # replace models GRUForecaster or TransformerForecaster
        input_size=6,
        hidden_size=32, #32
        n_heads=8, # Used by Transformer only
        hidden_fc=32, #128
        n_layers=3, # Encoder Layers
        window_size=window_size - forecast_horizon,
        forecast_horizon=forecast_horizon,
        conv_channels=16,
        kernel_size=9,
        dropout=0.1,
        use_pos_encoding = True # recommended to keep at True
    ).to(device)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr = learning_rate)
print(f"Created {type(model).__name__}, Using Positional Encoding:{bool(model.pos_encoding)}")
import torchinfo

# Display model summary and parameter count
torchinfo.summary(model, input_size=(BATCH_SIZE, window_size - forecast_horizon, model.input_proj.in_features), device=device)

Created TransformerForecaster, Using Positional Encoding:True


Layer (type:depth-idx)                        Output Shape              Param #
TransformerForecaster                         [64, 1, 6]                --
├─Linear: 1-1                                 [64, 199, 32]             224
├─PositionalEncoding: 1-2                     [64, 199, 32]             --
├─TransformerEncoderStack: 1-3                [64, 199, 32]             --
│    └─ModuleList: 2-1                        --                        --
│    │    └─TransformerEncoderLayer: 3-1      [64, 199, 32]             6,464
│    │    └─TransformerEncoderLayer: 3-2      [64, 199, 32]             6,464
│    │    └─TransformerEncoderLayer: 3-3      [64, 199, 32]             6,464
├─ForecastDecoder: 1-4                        [64, 1, 6]                --
│    └─Conv1d: 2-2                            [64, 16, 199]             13,840
│    └─ReLU: 2-3                              [64, 16, 199]             --
│    └─Sequential: 2-4                        [64, 6]                   --
│    │

#Training and Validation Loop

In [ ]:
epochs = 20
train_loss_history = []
val_loss_history = []
total_time = 0
# --- train ---
for epoch in range(epochs):
    start_time = time.perf_counter()
    model.train()
    batch_losses = []
    for batch in train_loader:

        x_data = batch[0].to(device)

        inputs = x_data[:, :-forecast_horizon, :]
        targets = x_data[:, -forecast_horizon:, :]

        optimizer.zero_grad()
        y_pred = model(inputs)
        loss = criterion(y_pred, targets)
        loss.backward()
        optimizer.step()
        batch_losses.append(loss.item())
    train_loss = np.mean(batch_losses)
    end_time = time.perf_counter()
    total_time += (end_time - start_time)


#---validate---
    # --- Step A: Calibrate the threshold on the held-out VALIDATION set
    #     (normal-only, never seen during training and never part of the test
    #     metrics below). ---
    model.eval()
    val_losses = []
    val_scores = []
    with torch.no_grad():
        for batch in val_loader:
            x_data = batch[0].to(device)
            inputs = x_data[:, :-forecast_horizon, :]
            targets = x_data[:, -forecast_horizon:, :]
            preds = model(inputs)
            sample_mse = torch.mean((preds - targets) ** 2, dim=(1, 2)).cpu().numpy()
            val_scores.extend(sample_mse)
            val_losses.append(criterion(preds, targets).item())
    val_scores = np.array(val_scores)
    threshold = np.percentile(val_scores, 95)  # 95% specificity, calibrated on held-out normal data
    val_loss = np.mean(val_losses)

    train_loss_history.append(train_loss)
    val_loss_history.append(val_loss)
    print(f"Epoch {epoch:>3}: train MSE loss = {train_loss:.6f}, val MSE loss = {val_loss:.6f}, train time: {(end_time - start_time):.3f}s")
print(f"Total Training Time for {type(model).__name__} is {total_time:.3f}s")

Epoch   0: train MSE loss = 0.406438, val MSE loss = 0.155239, train time: 1.419s
Epoch   1: train MSE loss = 0.175603, val MSE loss = 0.093335, train time: 1.064s


KeyboardInterrupt: 

#Val Loop
##We want to see how well the model predicts the unseen benign data. Ideally  the model does a good job generalizing benign data

In [ ]:
model.eval()

all_x_data_test = []
all_y_pred_test = []

with torch.no_grad():
    for batch_idx, batch in enumerate(val_loader):
        x_data = batch[0].to(device)

        inputs = x_data[:, :-forecast_horizon, :]
        y_pred = model(inputs)

        all_x_data_test.append(x_data.cpu().numpy())
        all_y_pred_test.append(y_pred.cpu().numpy())

# Concatenate all batches
all_x_data_test = np.concatenate(all_x_data_test, axis=0)
all_y_pred_test = np.concatenate(all_y_pred_test, axis=0)

num_samples_to_plot = 5
num_channels = all_x_data_test.shape[2] # Get number of channels

# Randomly select 5 indices to plot
if len(all_x_data_test) >= num_samples_to_plot:
    plot_indices = np.random.choice(len(all_x_data_test), num_samples_to_plot, replace=False)
else:
    plot_indices = np.arange(len(all_x_data_test))
    num_samples_to_plot = len(all_x_data_test)

fig, axes = plt.subplots(num_samples_to_plot, 1, figsize=(15, 4 * num_samples_to_plot))
if num_samples_to_plot == 1:
    axes = [axes] # Ensure axes is iterable even for a single plot

zoom_window = 20 # Zoom into the last 20 samples of the input

for i, idx in enumerate(plot_indices):
    full_original_sequence = all_x_data_test[idx] # shape (window_size, num_channels)
    predicted_future = all_y_pred_test[idx]       # shape (forecast_horizon, num_channels)

    input_part = full_original_sequence[:-forecast_horizon, :]
    actual_future_part = full_original_sequence[-forecast_horizon:, :]

    # Determine the start index for the zoom window
    start_idx = max(0, input_part.shape[0] - zoom_window)

    # Time steps for plotting
    time_steps_input = np.arange(start_idx, input_part.shape[0])
    time_steps_future = np.arange(input_part.shape[0], input_part.shape[0] + forecast_horizon)

    for channel in range(num_channels):
        # Plot input part and grab its color
        line = axes[i].plot(time_steps_input, input_part[start_idx:, channel], label=f'Input Ch {channel+1}', linestyle='-', alpha=0.7)
        color = line[0].get_color()

        # Plot actual future part matching the color
        axes[i].plot(time_steps_future, actual_future_part[:, channel], label=f'Actual Ch {channel+1}', linestyle='--', marker='o', markersize=5, color=color)

        # Plot predicted future part matching the color
        axes[i].plot(time_steps_future, predicted_future[:, channel], label=f'Pred Ch {channel+1}', linestyle=':', marker='x', markersize=7, markeredgewidth=2, color=color)

    axes[i].set_title(f'Sample {idx+1}: Predicted vs Actual Forecast (Last {zoom_window} Steps)', fontsize=14)
    axes[i].set_xlabel('Time Step', fontsize=12)
    axes[i].set_ylabel('Normalized Value', fontsize=12)
    axes[i].legend(loc='center left', bbox_to_anchor=(1.01, 0.5), fontsize=9, ncol=2)
    axes[i].grid(True)

plt.tight_layout()
plt.show()

# Test Loop Validation

In [ ]:
# --- Step B: Score the untouched test set ---
anomaly_scores = []
ground_truth = []

with torch.no_grad():
    for x_batch, y_batch in test_loader:
        x_batch = x_batch.to(device)
        inputs = x_batch[:, :-forecast_horizon, :]
        targets = x_batch[:, -forecast_horizon:, :]

        preds = model(inputs)
        sample_mse = torch.mean((preds - targets) ** 2, dim=(1, 2)).cpu().numpy()

        anomaly_scores.extend(sample_mse)
        ground_truth.extend(y_batch.numpy())

anomaly_scores = np.array(anomaly_scores)
ground_truth = np.array(ground_truth)

binary_preds = (anomaly_scores >= threshold).astype(int)

tn, fp, fn, tp = confusion_matrix(ground_truth, binary_preds).ravel()
far = fp / (fp + tn)
precision = precision_score(ground_truth, binary_preds, zero_division=0)
recall = recall_score(ground_truth, binary_preds, zero_division=0)
f1 = f1_score(ground_truth, binary_preds, zero_division=0)
roc_auc = roc_auc_score(ground_truth, anomaly_scores)

print("==================================================")
print("           UNIFIED EVALUATION REPORT              ")
print("==================================================")
print(f"Chosen Decision Threshold : {threshold:.6f}  (calibrated on held-out val set)")
print("--------------------------------------------------")
print(f"False Alarm Rate (FAR)    : {far * 100:.2f}%")
print(f"Precision                 : {precision:.4f}")
print(f"Recall (Sensitivity)      : {recall:.4f}")
print(f"F1-Score                  : {f1:.4f}")
print(f"ROC-AUC Score             : {roc_auc:.4f}")
print("--------------------------------------------------")
print("Confusion Matrix:")
print(f"  [TN: {tn:5d} | FP: {fp:5d}]  (Normal - UCI HAR)")
print(f"  [FN: {fn:5d} | TP: {tp:5d}]  (Anomaly - SisFall)")
print("==================================================")

#Isolation Forest Model (Classical Baseline)

### Unlike the GRU/Transformer forecasters above, IsolationForest doesn't work on raw sequences, so each already-standardized window (same `X_fit_tensor` / `X_val_tensor` / `X_test_tensor` from the data prep section) is condensed into a flat vector of per-channel summary statistics plus accel/gyro magnitude statistics. Threshold calibration (95th percentile on the held-out val set) and the evaluation report exactly mirror the GRU/Transformer sections above, so all three models are directly comparable.

In [ ]:
from sklearn.ensemble import IsolationForest

# --- Feature engineering for the classical model ---
# Raw 200-step x 6-channel sequences are too high-dimensional / unstructured
# for a tree-based method like IsolationForest, so each window is condensed
# into a flat feature vector. Inputs here are the SAME standardized tensors
# (X_fit_tensor / X_val_tensor / X_test_tensor) the GRU/Transformer models
# use, so this is an apples-to-apples comparison on identical data/splits.
def extract_window_features(x_tensor):
    x = x_tensor.numpy()  # (N, T, C), already z-scored using fit_mean/fit_std

    accel = x[:, :, 0:3]
    gyro = x[:, :, 3:6]
    accel_mag = np.linalg.norm(accel, axis=2)  # (N, T)
    gyro_mag = np.linalg.norm(gyro, axis=2)    # (N, T)

    # Per-channel summary stats: mean, std, min, max, RMS -> (N, 6*5=30)
    per_channel_feats = np.concatenate([
        x.mean(axis=1),
        x.std(axis=1),
        x.min(axis=1),
        x.max(axis=1),
        np.sqrt((x ** 2).mean(axis=1)),
    ], axis=1)

    # Accel/gyro magnitude stats -> (N, 6)
    magnitude_feats = np.stack([
        accel_mag.mean(axis=1), accel_mag.std(axis=1), accel_mag.max(axis=1),
        gyro_mag.mean(axis=1), gyro_mag.std(axis=1), gyro_mag.max(axis=1),
    ], axis=1)

    return np.concatenate([per_channel_feats, magnitude_feats], axis=1)  # (N, 36)

X_fit_feats = extract_window_features(X_fit_tensor)
X_val_feats = extract_window_features(X_val_tensor)
X_test_feats = extract_window_features(X_test_tensor)

# --- Fit Isolation Forest on normal-only training features ---
# (mirrors train_loader: fit only on the benign fit pool, never on anomalies)
iso_forest = IsolationForest(
    n_estimators=200,
    max_samples="auto",
    contamination="auto",  # we calibrate our own decision threshold below instead
    random_state=0,
    n_jobs=-1,
)
iso_forest.fit(X_fit_feats)
print(f"Isolation Forest fit on {X_fit_feats.shape[0]} normal windows, {X_fit_feats.shape[1]} features/window")

# --- Step A (mirrors cell 19's val loop): calibrate threshold on held-out val set ---
# score_samples() is HIGHER for inliers, so we negate it -> higher = more anomalous,
# matching the MSE-based anomaly_scores convention used by the GRU/Transformer.
val_scores_if = -iso_forest.score_samples(X_val_feats)
threshold_if = np.percentile(val_scores_if, 95)  # 95% specificity, same rule as above

# --- Step B (mirrors cell 23): score the untouched test set ---
anomaly_scores_if = -iso_forest.score_samples(X_test_feats)
ground_truth_if = y_test_tensor.numpy()

binary_preds_if = (anomaly_scores_if >= threshold_if).astype(int)

tn, fp, fn, tp = confusion_matrix(ground_truth_if, binary_preds_if).ravel()
far = fp / (fp + tn)
precision = precision_score(ground_truth_if, binary_preds_if, zero_division=0)
recall = recall_score(ground_truth_if, binary_preds_if, zero_division=0)
f1 = f1_score(ground_truth_if, binary_preds_if, zero_division=0)
roc_auc = roc_auc_score(ground_truth_if, anomaly_scores_if)

print("==================================================")
print("     UNIFIED EVALUATION REPORT (Isolation Forest)  ")
print("==================================================")
print(f"Chosen Decision Threshold : {threshold_if:.6f}  (calibrated on held-out val set)")
print("--------------------------------------------------")
print(f"False Alarm Rate (FAR)    : {far * 100:.2f}%")
print(f"Precision                 : {precision:.4f}")
print(f"Recall (Sensitivity)      : {recall:.4f}")
print(f"F1-Score                  : {f1:.4f}")
print(f"ROC-AUC Score             : {roc_auc:.4f}")
print("--------------------------------------------------")
print("Confusion Matrix:")
print(f"  [TN: {tn:5d} | FP: {fp:5d}]  (Normal - UCI HAR)")
print(f"  [FN: {fn:5d} | TP: {tp:5d}]  (Anomaly - SisFall)")
print("==================================================")

## [EXTRA ANALYSIS 1] Isolation Forest: Score Separation & ROC/PR Curves

**What it does:** Plots the anomaly score histogram (normal vs. anomaly) with the threshold marked, plus ROC and Precision-Recall curves.
**Why:** Shows performance across *all* possible thresholds, not just the one 95th-percentile cutoff used in the report above — useful if you want to argue the threshold choice, or show how forgiving/strict a different cutoff would be.

In [ ]:
from sklearn.metrics import roc_curve

# --- How well-separated are normal vs. anomaly scores, independent of
# the single 95th-percentile threshold chosen above? ---
normal_scores_if = anomaly_scores_if[ground_truth_if == 0]
attack_scores_if = anomaly_scores_if[ground_truth_if == 1]

fpr, tpr, _ = roc_curve(ground_truth_if, anomaly_scores_if)
prec_curve, rec_curve, _ = precision_recall_curve(ground_truth_if, anomaly_scores_if)
pr_auc = auc(rec_curve, prec_curve)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Score distribution: normal vs anomaly, with the chosen threshold marked
axes[0].hist(normal_scores_if, bins=40, alpha=0.6, label="Normal (UCI HAR)", color="tab:blue")
axes[0].hist(attack_scores_if, bins=40, alpha=0.6, label="Anomaly (SisFall)", color="tab:red")
axes[0].axvline(threshold_if, color="black", linestyle="--", label=f"Threshold ({threshold_if:.3f})")
axes[0].set_xlabel("Anomaly Score (-score_samples)")
axes[0].set_ylabel("Count")
axes[0].set_title("Isolation Forest: Test Score Distribution")
axes[0].legend()

# 2. ROC curve
axes[1].plot(fpr, tpr, color="tab:purple", label=f"ROC-AUC = {roc_auc:.4f}")
axes[1].plot([0, 1], [0, 1], linestyle="--", color="gray", label="Chance")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("ROC Curve")
axes[1].legend()

# 3. Precision-Recall curve (more informative than ROC given the 20% anomaly rate)
axes[2].plot(rec_curve, prec_curve, color="tab:green", label=f"PR-AUC = {pr_auc:.4f}")
axes[2].set_xlabel("Recall")
axes[2].set_ylabel("Precision")
axes[2].set_title("Precision-Recall Curve")
axes[2].legend()

plt.tight_layout()
plt.show()

print(f"Mean anomaly score - Normal   : {normal_scores_if.mean():.4f} (+/- {normal_scores_if.std():.4f})")
print(f"Mean anomaly score - Anomaly  : {attack_scores_if.mean():.4f} (+/- {attack_scores_if.std():.4f})")
print(f"PR-AUC (threshold-independent): {pr_auc:.4f}")

## [EXTRA ANALYSIS 2] Isolation Forest: Feature Importance (Permutation-Based)

**What it does:** Shuffles one engineered feature at a time across the test set and measures how much ROC-AUC drops.
**Why:** IsolationForest has no built-in `feature_importances_` like a supervised Random Forest. This is the standard workaround, and it also gives interpretability the GRU/Transformer black-boxes can't offer — you can say *which* signal statistics actually drove detection.

In [ ]:
# --- Permutation importance via ROC-AUC degradation ---
# For each engineered feature, shuffle it across the test set and see how much
# ROC-AUC drops relative to the unpermuted baseline. Repeated multiple times
# per feature (different shuffles) and averaged, for stability.
feature_names = (
    [f"{stat}_{ch}" for stat in ["mean", "std", "min", "max", "rms"]
     for ch in ["accel_x", "accel_y", "accel_z", "gyro_x", "gyro_y", "gyro_z"]]
    + ["accel_mag_mean", "accel_mag_std", "accel_mag_max",
       "gyro_mag_mean", "gyro_mag_std", "gyro_mag_max"]
)
assert len(feature_names) == X_test_feats.shape[1]

baseline_auc = roc_auc_score(ground_truth_if, anomaly_scores_if)

n_repeats = 10
rng = np.random.RandomState(0)  # local RNG, doesn't touch the notebook's global seed state
importances = np.zeros((n_repeats, X_test_feats.shape[1]))

for j in range(X_test_feats.shape[1]):
    for r in range(n_repeats):
        X_permuted = X_test_feats.copy()
        X_permuted[:, j] = rng.permutation(X_permuted[:, j])
        permuted_scores = -iso_forest.score_samples(X_permuted)
        permuted_auc = roc_auc_score(ground_truth_if, permuted_scores)
        importances[r, j] = baseline_auc - permuted_auc

mean_importance = importances.mean(axis=0)
std_importance = importances.std(axis=0)

order = np.argsort(mean_importance)[::-1][:15]  # top 15 features

plt.figure(figsize=(9, 6))
plt.barh([feature_names[i] for i in order][::-1],
         mean_importance[order][::-1],
         xerr=std_importance[order][::-1],
         color="tab:orange")
plt.xlabel("Mean ROC-AUC Drop When Shuffled")
plt.title(f"Isolation Forest: Top 15 Features by Permutation Importance\n(Baseline ROC-AUC = {baseline_auc:.4f})")
plt.tight_layout()
plt.show()

print("Top 5 most important features:")
for i in order[:5]:
    print(f"  {feature_names[i]:<16s}: AUC drop = {mean_importance[i]:.4f} (+/- {std_importance[i]:.4f})")

## [EXTRA ANALYSIS 3] Isolation Forest: Model Complexity

**What it does:** Reports ensemble size, tree depth, and node counts.
**Why:** The GRU/Transformer sections report parameter counts via `torchinfo`. IsolationForest has no learnable parameters, so this is the closest fair "model complexity" comparison point for your writeup.

In [ ]:
# --- Model complexity readout (Isolation Forest analog to torchinfo.summary) ---
tree_depths = [est.get_depth() for est in iso_forest.estimators_]
tree_node_counts = [est.tree_.node_count for est in iso_forest.estimators_]

print("==================================================")
print("     ISOLATION FOREST MODEL COMPLEXITY             ")
print("==================================================")
print(f"Number of Trees        : {len(iso_forest.estimators_)}")
print(f"Samples per Tree       : {iso_forest.max_samples_}")
print(f"Input Feature Count    : {X_fit_feats.shape[1]}")
print(f"Avg. Tree Depth        : {np.mean(tree_depths):.2f} (+/- {np.std(tree_depths):.2f})")
print(f"Max Tree Depth         : {np.max(tree_depths)}")
print(f"Total Nodes (all trees): {np.sum(tree_node_counts)}")
print(f"Avg. Nodes per Tree    : {np.mean(tree_node_counts):.1f}")
print("==================================================")

## [EXTRA ANALYSIS 4] Isolation Forest: Perfect-Score Sanity Check

**What it does:** Ranks every feature by its *standalone* AUC (ignoring the rest of the ensemble), and plots the raw physical-unit distribution of the single most-separating feature by class.
**Why:** Recall and ROC-AUC both hit exactly 1.0000, and permutation importance above showed ~0.0000 AUC drop for every single feature — meaning many features are individually near-sufficient on their own. This check tests whether the model is really learning a subtle multivariate signature, or just exploiting a simple amplitude gap between the two source datasets (SisFall fall-impact spikes vs. UCI HAR daily activity). Worth a sentence in your writeup either way: this baseline may look artificially strong on this benchmark and might not generalize as well to subtler real damage events.

In [ ]:
# --- Single-feature AUC ranking (direction-agnostic) ---
# If any one engineered feature alone nearly separates the classes, the
# ensemble's near-perfect score is more likely explained by a dataset-level
# amplitude/scale gap than by a genuinely learned multivariate pattern.
single_feature_auc = np.array([
    max(roc_auc_score(ground_truth_if, X_test_feats[:, j]),
        1 - roc_auc_score(ground_truth_if, X_test_feats[:, j]))
    for j in range(X_test_feats.shape[1])
])
order3 = np.argsort(single_feature_auc)[::-1]

print("Top 10 features by STANDALONE discriminative power (direction-agnostic AUC):")
for i in order3[:10]:
    print(f"  {feature_names[i]:<16s}: single-feature AUC = {single_feature_auc[i]:.4f}")

plt.figure(figsize=(9, 5))
plt.barh([feature_names[i] for i in order3[:15]][::-1], single_feature_auc[order3[:15]][::-1], color="tab:red")
plt.axvline(0.5, color="gray", linestyle="--", label="Chance (0.5)")
plt.xlabel("Standalone ROC-AUC (direction-agnostic)")
plt.title("Isolation Forest: How Much Does ONE Feature Alone Separate the Classes?")
plt.legend()
plt.tight_layout()
plt.show()

# --- Visual sanity check in RAW physical units (undo standardization) ---
top_feat_idx = order3[0]
X_test_raw = X_test_tensor * fit_std + fit_mean          # back to physical units
X_test_feats_raw = extract_window_features(X_test_raw)  # same engineered features, raw scale

normal_vals = X_test_feats_raw[ground_truth_if == 0, top_feat_idx]
anomaly_vals = X_test_feats_raw[ground_truth_if == 1, top_feat_idx]

plt.figure(figsize=(6, 5))
plt.boxplot([normal_vals, anomaly_vals], labels=["Normal (UCI HAR)", "Anomaly (SisFall)"])
plt.ylabel(f"{feature_names[top_feat_idx]} (raw physical units)")
plt.title(f"Top Single Feature by Class: '{feature_names[top_feat_idx]}'")
plt.tight_layout()
plt.show()

print(f"\nTop feature '{feature_names[top_feat_idx]}' standalone AUC: {single_feature_auc[top_feat_idx]:.4f}")
print(f"  Normal  median: {np.median(normal_vals):.4f}")
print(f"  Anomaly median: {np.median(anomaly_vals):.4f}")

# 20% Anomaly Results

## Vanilla Transformer

```
==================================================
           UNIFIED EVALUATION REPORT
==================================================
Chosen Decision Threshold : 0.176646  (calibrated on held-out val set)
--------------------------------------------------
False Alarm Rate (FAR)    : 4.82%
Precision                 : 0.7617
Recall (Sensitivity)      : 0.6168
F1-Score                  : 0.6817
ROC-AUC Score             : 0.9047
--------------------------------------------------
Confusion Matrix:
  [TN:  2805 | FP:   142]  (Normal - UCI HAR)
  [FN:   282 | TP:   454]  (Anomaly - SisFall)
==================================================
```



## GRU-CNN Results


```
==================================================
           UNIFIED EVALUATION REPORT
==================================================
Chosen Decision Threshold : 0.132007  (calibrated on held-out val set)
--------------------------------------------------
False Alarm Rate (FAR)    : 5.16%
Precision                 : 0.7951
Recall (Sensitivity)      : 0.8016
F1-Score                  : 0.7984
ROC-AUC Score             : 0.9643
--------------------------------------------------
Confusion Matrix:
  [TN:  2795 | FP:   152]  (Normal - UCI HAR)
  [FN:   146 | TP:   590]  (Anomaly - SisFall)
==================================================
```



## Transformer-CNN Results

```
==================================================
           UNIFIED EVALUATION REPORT
==================================================
Chosen Decision Threshold : 0.183792  (calibrated on held-out val set)
--------------------------------------------------
False Alarm Rate (FAR)    : 5.60%
Precision                 : 0.7911
Recall (Sensitivity)      : 0.8492
F1-Score                  : 0.8191
ROC-AUC Score             : 0.9678
--------------------------------------------------
Confusion Matrix:
  [TN:  2782 | FP:   165]  (Normal - UCI HAR)
  [FN:   111 | TP:   625]  (Anomaly - SisFall)
==================================================
```



## Isolation Forest (Classical Baseline)

<pre>
==================================================
     UNIFIED EVALUATION REPORT (Isolation Forest)  
==================================================
Chosen Decision Threshold : 0.544022  (calibrated on held-out val set)
--------------------------------------------------
False Alarm Rate (FAR)    : 3.87%
Precision                 : 0.8659
Recall (Sensitivity)      : 1.0000
F1-Score                  : 0.9281
ROC-AUC Score             : 1.0000
--------------------------------------------------
Confusion Matrix:
  [TN:  2833 | FP:   114]  (Normal - UCI HAR)
  [FN:     0 | TP:   736]  (Anomaly - SisFall)
==================================================
</pre>

# 50% Anomaly Results

## Transformer-CNN Results
```==================================================
           UNIFIED EVALUATION REPORT              
==================================================
Chosen Decision Threshold : 0.149196  (calibrated on held-out val set)
--------------------------------------------------
False Alarm Rate (FAR)    : 5.33%
Precision                 : 0.9425
Recall (Sensitivity)      : 0.8744
F1-Score                  : 0.9072
ROC-AUC Score             : 0.9752
--------------------------------------------------
Confusion Matrix:
  [TN:  1651 | FP:    93]  (Normal - UCI HAR)
  [FN:   219 | TP:  1525]  (Anomaly - SisFall)
==================================================

## GRU-CNN Results
```==================================================
           UNIFIED EVALUATION REPORT              
==================================================
Chosen Decision Threshold : 0.162932  (calibrated on held-out val set)
--------------------------------------------------
False Alarm Rate (FAR)    : 3.33%
Precision                 : 0.9608
Recall (Sensitivity)      : 0.8148
F1-Score                  : 0.8818
ROC-AUC Score             : 0.9737
--------------------------------------------------
Confusion Matrix:
  [TN:  1686 | FP:    58]  (Normal - UCI HAR)
  [FN:   323 | TP:  1421]  (Anomaly - SisFall)
==================================================

## Isolation Forest Results
```Isolation Forest fit on 6617 normal windows, 36 features/window
==================================================
     UNIFIED EVALUATION REPORT (Isolation Forest)  
==================================================
Chosen Decision Threshold : 0.538525  (calibrated on held-out val set)
--------------------------------------------------
False Alarm Rate (FAR)    : 3.96%
Precision                 : 0.9619
Recall (Sensitivity)      : 1.0000
F1-Score                  : 0.9806
ROC-AUC Score             : 1.0000
--------------------------------------------------
Confusion Matrix:
  [TN:  1675 | FP:    69]  (Normal - UCI HAR)
  [FN:     0 | TP:  1744]  (Anomaly - SisFall)
==================================================